# Gold — dim_causa

UNION DISTINCT de `(codigo_origen, descripcion)` desde todas las fuentes Silver.
Unifica CIE-10 (INE, MSPAS), rangos de causas externas (INE) y códigos OMS (WHO).

| Columna | Tipo | Descripción |
|---|---|---|
| `causa_sk` | INT | Surrogate key |
| `codigo_origen` | STRING | Mejor código disponible por fuente (CIE-10 normalizado o código OMS) |
| `descripcion` | STRING | Nombre de la causa de muerte o diagnóstico |

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from functools import reduce
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

SILVER_SCHEMA = 'stage_silver_ss2'
GOLD_SCHEMA   = 'gold_ss2'
WRITE_MODE    = 'overwrite'

def get_job_run_id():
    try:
        return (
            dbutils.notebook.entry_point
            .getDbutils().notebook().getContext()
            .currentRunId().toString()
        )
    except Exception:
        return 'manual'

RUN_ID = get_job_run_id()
logger.info(f'Setup OK — run_id={RUN_ID}')

## INE-edad (3 tablas)

Columna fuente: `cie_10_norm` → `codigo_origen`, `causa_de_muerte` → `descripcion`.

In [ ]:
_INE_EDAD = [
    'ine_defunciones_sexo_edad',
    'ine_defunciones_neonatales',
    'ine_defunciones_postneonatales',
]

sources = []

for t in _INE_EDAD:
    df = (
        spark.table(f'{SILVER_SCHEMA}.{t}')
        .select(
            F.col('cie_10_norm').alias('codigo_origen'),
            F.col('causa_de_muerte').alias('descripcion')
        )
    )
    sources.append(df)
    logger.info(f'[{t}] causas distintas: {df.dropDuplicates().count()}')

## INE-geo residencia

Columna fuente: `cie_10_norm` → `codigo_origen`, `causa_de_muerte` → `descripcion`.

In [ ]:
df_ine_geo = (
    spark.table(f'{SILVER_SCHEMA}.ine_defunciones_depto_residencia')
    .select(
        F.col('cie_10_norm').alias('codigo_origen'),
        F.col('causa_de_muerte').alias('descripcion')
    )
)
sources.append(df_ine_geo)
logger.info(f'[ine_defunciones_depto_residencia] causas distintas: {df_ine_geo.dropDuplicates().count()}')

## INE causas externas

Columna fuente: `cie_10_norm` → `codigo_origen`, `causa_de_muerte` → `descripcion`.
Los valores son rangos de texto normalizados (ej. `Accidentesdetransporte(V01-V99)`).

In [ ]:
df_causas_ext = (
    spark.table(f'{SILVER_SCHEMA}.ine_defunciones_causas_externas')
    .select(
        F.col('cie_10_norm').alias('codigo_origen'),
        F.col('causa_de_muerte').alias('descripcion')
    )
)
sources.append(df_causas_ext)
logger.info(f'[ine_defunciones_causas_externas] causas distintas: {df_causas_ext.dropDuplicates().count()}')

## MSPAS (3 tablas)

Columna fuente: `cie_10_norm` → `codigo_origen`, `diagnostico` → `descripcion`.

In [ ]:
_MSPAS = [
    'dbx_primeras_causas_de_morbilidad_2015_a_2025',
    'dbx_morbilidad_enfermedades_cronicas_2015_a_2025',
    'dbx_morbilidad_grupo_materno_infantil_2012_a_2025',
]

for t in _MSPAS:
    df = (
        spark.table(f'{SILVER_SCHEMA}.{t}')
        .select(
            F.col('cie_10_norm').alias('codigo_origen'),
            F.col('diagnostico').alias('descripcion')
        )
    )
    sources.append(df)
    logger.info(f'[{t}] causas distintas: {df.dropDuplicates().count()}')

## WHO (2 tablas)

Columna fuente: `codigo_causa` → `codigo_origen`, `nombre_causa` → `descripcion`.
Los códigos OMS (ej. `CG0395`) no son comparables con CIE-10.

In [ ]:
_WHO = ['who_guatemala', 'who_costa_rica']

for t in _WHO:
    df = (
        spark.table(f'{SILVER_SCHEMA}.{t}')
        .select(
            F.col('codigo_causa').alias('codigo_origen'),
            F.col('nombre_causa').alias('descripcion')
        )
    )
    sources.append(df)
    logger.info(f'[{t}] causas distintas: {df.dropDuplicates().count()}')

## UNION y escritura

In [ ]:
df_union = (
    reduce(lambda a, b: a.union(b), sources)
    .filter(F.col('codigo_origen').isNotNull())
    .dropDuplicates(['codigo_origen', 'descripcion'])
)

df_dim = df_union.withColumn(
    'causa_sk',
    F.row_number().over(Window.orderBy('codigo_origen'))
)

df_dim = df_dim.select('causa_sk', 'codigo_origen', 'descripcion')

logger.info(f'dim_causa: {df_dim.count()} causas únicas')

df_dim.write \
    .format('delta') \
    .mode(WRITE_MODE) \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{GOLD_SCHEMA}.dim_causa')

logger.info(f'Escrito → {GOLD_SCHEMA}.dim_causa')

## Validación

In [ ]:
df_val = spark.table(f'{GOLD_SCHEMA}.dim_causa')
print(f'Total causas: {df_val.count()}')
print('\n── Muestra ──')
df_val.limit(15).show(truncate=False)

print('\n── Verificar NULLs en codigo_origen ──')
null_count = df_val.filter(F.col('codigo_origen').isNull()).count()
print(f'Registros con codigo_origen NULL: {null_count} (debe ser 0)')